# Usage Profiles: k-means Clustering

Segmentation of service users by how they engage with the service over time, using engagement features derived from a session log.

**All data in this notebook are simulated.** Feature distributions, sample size, the chosen number of clusters and the profile labels are invented for demonstration and do not correspond to any real dataset.

### Imports

In [ ]:
import numpy as np                      # numerical arrays and random numbers
import pandas as pd                     # tabular data (DataFrames)
import matplotlib.pyplot as plt         # plotting
import seaborn as sns                   # statistical plots on top of matplotlib
from sklearn.cluster import KMeans                  # clustering algorithm
from sklearn.decomposition import PCA               # dimensionality reduction (used only for plotting)
from sklearn.impute import SimpleImputer            # fills missing values
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score  # cluster-quality indices
from sklearn.preprocessing import StandardScaler    # standardises features to mean 0, sd 1

### Simulated session log

The simulated log covers a 36-month observation window (months 0 to 35). Each user is given a usage style that sets how many months they are active, how spread out those months are, and how many sessions they have per active month. Two rules of the programme are applied:

- **At most 12 sessions per year** per user, the maximum the programme covers.
- **Inclusion criterion:** only users whose first session is at least 12 months before the end of the window are kept, so every user has at least one year of possible activity.

In [ ]:
rng = np.random.default_rng(11)      # random generator with a fixed seed (reproducible results)

N_USERS = 7500                       # users simulated before applying the inclusion criterion
N_MONTHS = 36                        # observation window: months 0 to 35
MAX_SESSIONS_PER_YEAR = 12           # maximum number of sessions covered by the programme per year

rows = []
for user_id in range(N_USERS):

    # 1. Choose a usage style for this user
    style = rng.choice(["one-off", "short", "moderate", "long"], p=[0.4, 0.25, 0.2, 0.15])

    # 2. Settings of each style:
    #    n_active      = number of months with at least one session
    #    spread        = extra months over which those active months are scattered
    #    mean_sessions = average number of sessions in an active month
    #    (rng.integers(a, b) gives a whole number from a to b - 1)
    if style == "one-off":
        n_active = 1
        spread = 0
        mean_sessions = 1.3
    elif style == "short":
        n_active = rng.integers(2, 4)
        spread = rng.integers(0, 3)
        mean_sessions = 3.0
    elif style == "moderate":
        n_active = rng.integers(3, 8)
        spread = rng.integers(2, 12)
        mean_sessions = 1.8
    else:  # "long"
        n_active = rng.integers(6, 13)
        spread = rng.integers(8, 24)
        mean_sessions = 1.5

    # 3. Month of the first session
    first_month = rng.integers(0, N_MONTHS)

    # 4. Choose the other active months among the months that follow the first one
    later_months = list(range(first_month + 1, first_month + n_active + spread))
    if n_active > 1:
        other_months = list(rng.choice(later_months, size=n_active - 1, replace=False))
    else:
        other_months = []
    active_months = sorted([first_month] + other_months)

    # 5. Number of sessions in each active month, respecting the yearly maximum
    sessions_used = {}                                   # sessions already used in each year
    for month in active_months:
        if month >= N_MONTHS:                            # month falls after the end of the window
            continue

        n_sessions = 1 + rng.poisson(mean_sessions - 1)  # at least 1 session in an active month

        year = month // 12                               # year 0, 1 or 2 of the window
        used = sessions_used.get(year, 0)
        sessions_left = MAX_SESSIONS_PER_YEAR - used
        if n_sessions > sessions_left:                   # cap at the programme maximum
            n_sessions = sessions_left
        if n_sessions == 0:                              # no sessions left this year
            continue

        sessions_used[year] = used + n_sessions
        rows.append({"user_id": user_id, "month": month, "n_sessions": n_sessions})

# Monthly log: one row per user per active month
monthly = pd.DataFrame(rows)

# Inclusion criterion: first session at least 12 months before the end of the window
first_session = monthly.groupby("user_id")["month"].min()
included_users = first_session[first_session <= N_MONTHS - 12].index
monthly = monthly[monthly["user_id"].isin(included_users)]

print("Users included:", monthly["user_id"].nunique())
monthly.head()

Quick check that the programme rules hold in the simulated log:

In [ ]:
# Total sessions per user per year must never exceed 12
sessions_per_year = monthly.groupby(["user_id", monthly["month"] // 12])["n_sessions"].sum()
print("Maximum sessions in one year:", sessions_per_year.max())

# Every included user must have at least 12 months of possible activity
months_observed = N_MONTHS - monthly.groupby("user_id")["month"].min()
print("Minimum months observed per user:", months_observed.min())

### Feature engineering

Sessions are aggregated into monthly engagement features for each user:

| Feature | Description |
| --- | --- |
| `active_months` | Distinct months with at least one session |
| `total_months` | Span in months between first and last session |
| `sessions_per_month` | Mean sessions in active months |
| `sessions_in_peak_month` | Maximum sessions in a single month |
| `gini_monthly` | Gini coefficient of the monthly distribution (unevenness of use) |

In [ ]:
def gini(x):
    """Gini coefficient of a user's monthly session counts.

    0 = sessions spread perfectly evenly across months; values closer to 1 =
    sessions concentrated in a few months. Undefined (NaN) with a single month.

    Formula: G = sum over all pairs (i, j) of |x_i - x_j| / (2 * n^2 * mean)
    i.e. the mean absolute difference between months, relative to the mean.
    """
    x = list(x)                       # monthly session counts of one user
    n = len(x)                        # number of active months
    if n < 2 or sum(x) == 0:
        return np.nan

    mean = sum(x) / n

    # Sum of the absolute differences between every pair of months
    total_difference = 0
    for i in range(n):
        for j in range(n):
            total_difference += abs(x[i] - x[j])

    return total_difference / (2 * n ** 2 * mean)



# Collapse the monthly log into one row per user with five engagement features
features = monthly.groupby("user_id").agg(
    active_months=("month", "nunique"),                          # distinct active months
    total_months=("month", lambda m: m.max() - m.min() + 1),     # span from first to last month (inclusive)
    sessions_per_month=("n_sessions", "mean"),                   # average intensity
    sessions_in_peak_month=("n_sessions", "max"),                # busiest month
    gini_monthly=("n_sessions", gini),                           # unevenness of use over time
)

# Summary statistics (count, mean, sd, quartiles) for each feature
features.describe().round(2)

### Preprocessing

Missing values (e.g. the Gini index is undefined for a single active month) are imputed with the median, and features are standardised so that each one contributes equally to the distances used by k-means.

In [ ]:
FEATURES = list(features.columns)

# Fill missing values with the column median (k-means cannot handle NaN)
df_imputed = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(features), columns=FEATURES)

# Convert every feature to z-scores: (x - mean) / sd.
# k-means uses Euclidean distance, so without scaling the features with large
# ranges (e.g. total_months) would dominate those with small ranges (e.g. Gini)
X_scaled = StandardScaler().fit_transform(df_imputed)

### Model selection

k-means is fitted for k = 2 to 6 (the largest number still considered interpretable) and each solution is scored with inertia (elbow), silhouette, Calinski–Harabasz and Davies–Bouldin.

In [ ]:
rows = []
for k in range(2, 7):
    # n_init=10: run k-means from 10 different random starts and keep the best,
    # to avoid getting stuck in a poor local solution
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)   # fit the model and get each user's cluster
    rows.append({
        "k": k,
        # Inertia: within-cluster sum of squared distances (lower = tighter;
        # always decreases with k, so we look for the "elbow")
        "inertia": km.inertia_,
        # Silhouette: how much closer points are to their own cluster than to
        # the nearest other cluster, from -1 to 1 (higher = better)
        "silhouette": silhouette_score(X_scaled, labels),
        # Calinski-Harabasz: between-cluster vs within-cluster variance (higher = better)
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        # Davies-Bouldin: average similarity of each cluster to its most similar one (lower = better)
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })
results = pd.DataFrame(rows).set_index("k")

# Elbow plot and silhouette plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(results.index, results["inertia"], "o-")
axes[0].set(title="Elbow method", xlabel="k", ylabel="Inertia (SSE)")
axes[1].plot(results.index, results["silhouette"], "o-", color="tab:red")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
plt.show()

results.round(3)

The indices are weighed together, giving priority to the silhouette and Calinski–Harabasz scores. Here the choice is automated with a simple rank average:

In [ ]:
# Rank each k on every index (rank 1 = best), respecting the direction of each index
ranks = pd.concat([
    results["silhouette"].rank(ascending=False),         # higher is better
    results["calinski_harabasz"].rank(ascending=False),  # higher is better
    results["davies_bouldin"].rank(),                    # lower is better
], axis=1)

# Silhouette and Calinski-Harabasz count double
weights = np.array([2, 2, 1])

# Weighted sum of ranks per k; the k with the lowest total is selected
best_k = int((ranks * weights).sum(axis=1).idxmin())
print("Selected k:", best_k)

### Cluster description

In [ ]:
# Fit the final model with the selected number of clusters
final = KMeans(n_clusters=best_k, random_state=42, n_init=10)

# Labels are attached to the unscaled table so profiles can be read in real units
df_imputed["cluster"] = final.fit_predict(X_scaled)

# Mean value of each feature per cluster, plus the number of users in each
summary = df_imputed.groupby("cluster")[FEATURES].mean()
summary["n_users"] = df_imputed["cluster"].value_counts().sort_index()
summary.round(2)

Each cluster was given a descriptive name after reviewing its feature profile in the table above (LLM-assisted naming):

In [ ]:
# Names chosen after reviewing each cluster profile in the summary table
names = {
    0: "Brief (one-off)",          # ~1 active month, ~1 session per month
    1: "Long-term",                # ~7 active months spread over ~16 months
    2: "Intensive short-term",     # ~2-3 months, ~3 sessions per month, even use
    3: "Irregular with peaks",     # ~3-4 months, uneven use with a busy peak month
}

# Attach the profile name to every user
df_imputed["profile"] = df_imputed["cluster"].map(names)
df_imputed["profile"].value_counts()

In [ ]:
# Project the 5 standardised features onto 2 principal components for plotting only
# (the clustering itself used all 5 features)
pcs = PCA(n_components=2).fit_transform(X_scaled)
plot_df = pd.DataFrame(pcs, columns=["PC1", "PC2"]).assign(profile=df_imputed["profile"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: users in PCA space coloured by profile (random sample of 2,000 to avoid overplotting)
sns.scatterplot(data=plot_df.sample(2000, random_state=0), x="PC1", y="PC2", hue="profile",
                palette="tab10", s=12, ax=axes[0])
axes[0].set_title("Clusters in PCA space")

# Right: number of users per profile, largest first
order = df_imputed["profile"].value_counts().index
sns.countplot(data=df_imputed, x="profile", order=order, hue="profile", palette="tab10", legend=False, ax=axes[1])
axes[1].set_title("Users per profile")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()